# GEM_01: Data Loading
## Creates an input file (gem_input_data.pkl) with all the modalities to input into the GEM

## 1: GEM Data Loading and Preparation

In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import os
import pickle
sc.settings.verbosity = 0

In [2]:
BASE     = "/Users/maisievarcoe/Desktop/Research_Project/Scripts/Integration/GEM/pyglpk/Data"
 
PATHS = {
    # Tier 1 data (from R)
    "master_table":  f"{BASE}/master_table_final.csv",
    "sc_bridge":     f"{BASE}/scrna_bridge.csv",
    "conf_metabs":   f"{BASE}/high_conf_metabolites.csv",
    "conf_paths":    f"{BASE}/high_conf_pathways.csv",
    "nmr":           f"{BASE}/nmr_matched.csv",
    "microbiome":    f"{BASE}/species_matched.csv",
 
    # Single-cell expression (from R export)
    "mean_expr":     f"{BASE}/mean_expr_per_celltype.csv",
    "mean_expr_high":f"{BASE}/mean_expr_high_sa.csv",
    "mean_expr_low": f"{BASE}/mean_expr_low_sa.csv",
    "adata":         f"{BASE}/adata_qc.h5ad",
}
 
# Availability check
print("File availability:")
for name, path in PATHS.items():
    status = "✓" if os.path.exists(path) else "✗ NOT FOUND"
    print(f"  {status}  {name}")

File availability:
  ✓  master_table
  ✓  sc_bridge
  ✓  conf_metabs
  ✓  conf_paths
  ✓  nmr
  ✓  microbiome
  ✓  mean_expr
  ✓  mean_expr_high
  ✓  mean_expr_low
  ✓  adata


### 1.1: Load Tier 1 Data

In [5]:
# --- Master table: split into metadata + pathways on load ---
master_table = pd.read_csv(PATHS["master_table"])
 
meta_cols = [
    'NMR_sample_id', 'eg_id', 'sample_short', 'sample_id',
    'sa_abundance', 'SA_status', 'pH_mean', 'EASI', 'TEWL_mean',
    'lesional', 'sample_site', 'age', 'sex',
    'weeks_since_last_TCS_or_TCI', 'days_since_any_topicals',
    'local_EASI_intensity', 's_aureus_culture_growth_viapath'
]
pathway_cols = [c for c in master_table.columns if c not in meta_cols]
  
# --- Linking and confidence files ---
sc_bridge   = pd.read_csv(PATHS["sc_bridge"])
conf_metabs = pd.read_csv(PATHS["conf_metabs"])
conf_paths  = pd.read_csv(PATHS["conf_paths"])
 
# --- NMR metabolites ---
nmr = (pd.read_csv(PATHS["nmr"])
       .rename(columns={"Unnamed: 0": "NMR_sample_id"})
       .set_index("NMR_sample_id"))
 
# --- Microbiome species ---
microbiome = (pd.read_csv(PATHS["microbiome"])
              .rename(columns={"Unnamed: 0": "NMR_sample_id"})
              .set_index("NMR_sample_id"))
 
print(f"master_table:        {master_table.shape}")
print(f"metadata:            {metadata.shape}")
print(f"pathways_from_master:{pathways_from_master.shape}")
print(f"nmr:                 {nmr.shape}")
print(f"microbiome:          {microbiome.shape}")
 

master_table:        (20, 267)
metadata:            (19, 16)
pathways_from_master:(20, 250)
nmr:                 (18, 38)
microbiome:          (18, 44)


### 1.2: Load Single-Cell Cell-Type Expression Data

In [6]:
mean_expr      = pd.read_csv(PATHS["mean_expr"],      index_col=0)
mean_expr_high = pd.read_csv(PATHS["mean_expr_high"], index_col=0)
mean_expr_low  = pd.read_csv(PATHS["mean_expr_low"],  index_col=0)
 
print(f"mean_expr:           {mean_expr.shape}   (genes × cell types)")
print(f"mean_expr_high:      {mean_expr_high.shape}")
print(f"mean_expr_low:       {mean_expr_low.shape}")
print(f"Cell types: {list(mean_expr.columns)}")
 

mean_expr:           (23017, 11)   (genes × cell types)
mean_expr_high:      (23017, 11)
mean_expr_low:       (23017, 11)
Cell types: ['IFE SB', 'OB', 'IFE B', 'KER2', 'T CELL', 'IFE C', 'KER1', 'MEL', 'B CELL', 'uHF SB', 'FIB']


### 1.3: Handle Metadata and Pathways

In [7]:
# 1. Build metadata ONCE from master_table
metadata = master_table[meta_cols].copy()

# Drop rows without NMR_sample_id (these cannot be aligned)
metadata = metadata.dropna(subset=["NMR_sample_id"])

# Set index to NMR_sample_id
metadata = metadata.set_index("NMR_sample_id")

# 2. Build pathways ONCE from master_table
pathways_from_master = master_table[pathway_cols].copy()

# Set pathways index to NMR_sample_id as well
pathways_from_master.index = master_table["NMR_sample_id"].astype(str)

# 3. Clean all indices consistently
def clean_index(df):
    df.index = (
        df.index.astype(str)
        .str.strip()
        .str.replace("\r", "", regex=False)
        .str.replace("\n", "", regex=False)
    )
    return df

metadata = clean_index(metadata)
pathways_from_master = clean_index(pathways_from_master)
nmr = clean_index(nmr)
microbiome = clean_index(microbiome)

# 4. Compute shared sample IDs AFTER cleaning
common_ids = sorted(
    set(metadata.index)
    & set(pathways_from_master.index)
    & set(nmr.index)
    & set(microbiome.index)
)

# 5. Subset all modalities to the shared IDs
metadata = metadata.loc[common_ids]
pathways_from_master = pathways_from_master.loc[common_ids]
nmr = nmr.loc[common_ids]
microbiome = microbiome.loc[common_ids]

print("Metadata shape:", metadata.shape)
print("Pathways shape:", pathways_from_master.shape)
print("NMR shape:", nmr.shape)
print("Microbiome shape:", microbiome.shape)
print("Shared IDs:", len(common_ids))

Metadata shape: (18, 16)
Pathways shape: (18, 250)
NMR shape: (18, 38)
Microbiome shape: (18, 44)
Shared IDs: 18


### 2.1: Assign S. aureus Status (Low <20%, High >50%, exclude 20-50%)

In [8]:
def assign_sa_status(sa):
    if pd.isna(sa):   return None
    elif sa < 20:     return 'Low'
    elif sa > 50:     return 'High'
    else:             return None  # middle excluded from GEM comparison
 
metadata['SA_status'] = metadata['sa_abundance'].apply(assign_sa_status)
 
print(metadata[['sa_abundance', 'SA_status']]
      .sort_values('sa_abundance')
      .to_string())
print(f"\nFinal SA groups:\n{metadata['SA_status'].value_counts(dropna=False)}")


                       sa_abundance SA_status
NMR_sample_id                                
GYECZ1197.BACK.LES          0.00000       Low
GYECZ1197.RACF.NONLES       2.54032       Low
GYECZ1177.LACF.LES         10.61848       Low
GYECZ1178.LPF.NONLES       16.85067       Low
GYECZ1191.LACF.LES         31.31374      None
GYECZ1191.ABDO.LES         49.32039      None
GYECZ1183.LACF.NONLES      56.34637      High
GYECZ1193.LACF.NONLES      57.30307      High
GYECZ1191.ABDO.NONLES      67.47226      High
GYECZ1199.RACF.LES         72.82623      High
GYECZ1183.LACF.LES         76.37217      High
GYECZ1178.LPF.LES          83.50533      High
GYECZ1193.LACF.LES         89.31229      High
GYECZ1190.BACK.LES         98.56070      High
GYECZ1187.RACF.NONLES     100.00000      High
GYECZ1187.RACF.LES        100.00000      High
GYECZ1199.BACK.NONLES     100.00000      High
GYECZ1190.LACF.LES        100.00000      High

Final SA groups:
SA_status
High    12
Low      4
None     2
Name: count, dtype:

### 2.2: Clean Confidence Files

In [9]:
conf_paths  = conf_paths.dropna(subset=['feature'])
conf_metabs = conf_metabs.dropna(subset=['feature'])
 
conf_paths['feature_clean'] = (conf_paths['feature']
    .str.replace('.', ' ', regex=False)
    .str.replace('  ', ' ')
    .str.strip())
 
print("High-confidence pathways:")
print(conf_paths[['feature', 'Estimate', 'rho', 'p_value']].to_string())

High-confidence pathways:
                                                                  feature  Estimate       rho   p_value
0                          ARGININE.SYN4.PWY..L.ornithine.biosynthesis.II -0.000091 -0.569615  0.027795
1   ASPASN.PWY..superpathway.of.L.aspartate.and.L.asparagine.biosynthesis -0.000103 -0.635973  0.026376
2         GLYCOGENSYNTH.PWY..glycogen.biosynthesis.I..from.ADP.D.Glucose. -0.000105 -0.882601  0.000294
3                 HEME.BIOSYNTHESIS.II.1..heme.b.biosynthesis.V..aerobic. -0.000084 -0.636338  0.014089
4                             P161.PWY..acetylene.degradation..anaerobic. -0.000071 -0.555333  0.020157
5                 PWY.1269..CMP.3.deoxy.D.manno.octulosonate.biosynthesis -0.000047 -0.658579  0.015129
6             PWY.5136..fatty.acid..beta..oxidation.II..plant.peroxisome. -0.000112 -0.601741  0.043558
7           PWY.6292..superpathway.of.L.cysteine.biosynthesis..mammalian. -0.000110 -0.643751  0.025649
8                                    P

### 2.3: Compute High/Low S. aureus Group Means for Pathways 

In [10]:
# force numeric before taking means

model_samples = metadata[metadata['SA_status'].isin(['High', 'Low'])].copy()

high_ids = model_samples[model_samples['SA_status'] == 'High'].index.values
low_ids  = model_samples[model_samples['SA_status'] == 'Low'].index.values

print(f"High SA: {len(high_ids)} samples")
print(f"Low SA:  {len(low_ids)} samples")

# Force numeric — drops any stray text columns silently
pathways_numeric = pathways_from_master.apply(pd.to_numeric, errors='coerce')

# Check how many columns survived
n_before = pathways_from_master.shape[1]
n_after  = pathways_numeric.notna().any().sum()
print(f"\nPathway columns: {n_before} total, {n_after} numeric")

# Spot-check what got dropped (these are the stray metadata columns)
dropped = pathways_from_master.columns[pathways_numeric.isna().all()].tolist()
if dropped:
    print(f"Non-numeric columns dropped: {dropped}")

high_pathways_mean = pathways_numeric.loc[high_ids].mean()
low_pathways_mean  = pathways_numeric.loc[low_ids].mean()

print(f"\nHigh SA pathway mean range: {high_pathways_mean.min():.4f} to {high_pathways_mean.max():.4f}")
print(f"Low SA pathway mean range:  {low_pathways_mean.min():.4f} to {low_pathways_mean.max():.4f}")
 

High SA: 12 samples
Low SA:  4 samples

Pathway columns: 250 total, 250 numeric

High SA pathway mean range: 0.0000 to 70.0000
Low SA pathway mean range:  0.0000 to 368.0000


## 3: Verification

In [11]:
print("=" * 60)
print("FULL DATA VERIFICATION")
print("=" * 60)
 
checks = {
    "Metadata":    metadata,
    "Pathways":    pathways_from_master,
    "NMR":         nmr,
    "Microbiome":  microbiome,
}
for name, df in checks.items():
    print(f"\n[{name}]")
    print(f"  Shape:     {df.shape}")
    print(f"  Any NaNs:  {df.isna().sum().sum()}")
    print(f"  Index[:3]: {df.index[:3].tolist()}")
 
print(f"\n[Single-cell expression]")
for name, df in [("mean_expr", mean_expr),
                 ("mean_expr_high", mean_expr_high),
                 ("mean_expr_low", mean_expr_low)]:
    print(f"\n  {name}:")
    print(f"    Shape:       {df.shape}  (expect 23017 × 11)")
    print(f"    Value range: {df.values.min():.3f} to {df.values.max():.3f}  (expect 0–3.5)")
    print(f"    Any NaNs:    {df.isna().sum().sum()}")
 
# Shared genes and cell types across expression matrices
shared_genes = (mean_expr.index
                .intersection(mean_expr_high.index)
                .intersection(mean_expr_low.index))
shared_cts   = sorted(set(mean_expr.columns)
                      .intersection(mean_expr_high.columns)
                      .intersection(mean_expr_low.columns))
print(f"\n  Shared genes:      {len(shared_genes):,}")
print(f"  Shared cell types: {shared_cts}")
 
print(f"\n{'=' * 60}")
print("READY FOR GEM" if len(shared_genes) > 0 else "⚠ CHECK ABOVE BEFORE PROCEEDING")
print("=" * 60)

FULL DATA VERIFICATION

[Metadata]
  Shape:     (18, 16)
  Any NaNs:  2
  Index[:3]: ['GYECZ1177.LACF.LES', 'GYECZ1178.LPF.LES', 'GYECZ1178.LPF.NONLES']

[Pathways]
  Shape:     (18, 250)
  Any NaNs:  0
  Index[:3]: ['GYECZ1177.LACF.LES', 'GYECZ1178.LPF.LES', 'GYECZ1178.LPF.NONLES']

[NMR]
  Shape:     (18, 38)
  Any NaNs:  0
  Index[:3]: ['GYECZ1177.LACF.LES', 'GYECZ1178.LPF.LES', 'GYECZ1178.LPF.NONLES']

[Microbiome]
  Shape:     (18, 44)
  Any NaNs:  0
  Index[:3]: ['GYECZ1177.LACF.LES', 'GYECZ1178.LPF.LES', 'GYECZ1178.LPF.NONLES']

[Single-cell expression]

  mean_expr:
    Shape:       (23017, 11)  (expect 23017 × 11)
    Value range: 0.000 to 5.963  (expect 0–3.5)
    Any NaNs:    0

  mean_expr_high:
    Shape:       (23017, 11)  (expect 23017 × 11)
    Value range: 0.000 to 6.130  (expect 0–3.5)
    Any NaNs:    0

  mean_expr_low:
    Shape:       (23017, 11)  (expect 23017 × 11)
    Value range: 0.000 to 6.089  (expect 0–3.5)
    Any NaNs:    0

  Shared genes:      23,017
  

## 4: Get Target Metabolite IDs that Match Human2 GEM Style

In [12]:

mets = pd.read_csv("/Users/maisievarcoe/Desktop/Research_Project/Scripts/Integration/GEM/pyglpk/Data/metabolites.tsv", sep="\t")

# The Human2 Metbaolite IDs file contains kegg IDs, so i will match my target metabolites to these IDs and then to the model IDs suited to the GEM
def find_by_kegg(kegg_id):
    return mets[mets['metKEGGID'] == kegg_id]

kegg_map = {
    "Succinic acid": "C00042",
    "Lactic acid": "C00186",
    "Proline": "C00148",
    "Lysine": "C00047",
    "Alanine": "C00041",
    "Tyrosine": "C00082",
    "Isoleucine": "C00407",
    "Tryptophan": "C00078",
    "Glucose": "C00031",
    "Arginine": "C00062"
}

metabolite_map = {}

for name, kegg in kegg_map.items():
    row = mets[mets["metKEGGID"] == kegg]
    if len(row) > 0:
        metabolite_map[name] = row["mets"].values[0]

metabolite_map


{'Succinic acid': 'MAM02943c',
 'Lactic acid': 'MAM02403c',
 'Proline': 'MAM02770c',
 'Lysine': 'MAM02426c',
 'Alanine': 'MAM01307c',
 'Tyrosine': 'MAM03101c',
 'Isoleucine': 'MAM02184c',
 'Tryptophan': 'MAM03089c',
 'Glucose': 'MAM01965c',
 'Arginine': 'MAM01365c'}

## 5: Package and Save all GEM-Ready Data

In [13]:


# Ensure metadata and pathways are restricted to the 18 shared samples
metadata = metadata.loc[common_ids]
pathways_from_master = pathways_from_master.loc[common_ids]

all_10_metabolites = [
    "Succinic acid",
    "Lactic acid",
    "Proline",
    "Lysine",
    "Alanine",
    "Tyrosine",
    "Isoleucine",
    "Tryptophan",
    "Glucose",
    "Arginine"
]

gem_data = {
    "metadata": metadata,
    "nmr": nmr,
    "microbiome": microbiome,
    "pathways": pathways_from_master,
    "mean_expr": mean_expr,
    "mean_expr_high": mean_expr_high,
    "mean_expr_low": mean_expr_low,
    "metabolite_map": metabolite_map,      # Human-GEM IDs
    "gem_metabolites": all_10_metabolites, # <-- ALL TEN INCLUDED
    "common_ids": common_ids
}

OUTPUT_PATH = f"{BASE}/gem_input_data.pkl"

with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(gem_data, f)

print(f"\nSaved GEM-ready data to:\n  {OUTPUT_PATH}")
print("Contents:")
for k, v in gem_data.items():
    if hasattr(v, 'shape'):
        print(f"  {k:20s} → shape {v.shape}")
    else:
        print(f"  {k:20s} → {type(v)}")



Saved GEM-ready data to:
  /Users/maisievarcoe/Desktop/Research_Project/Scripts/Integration/GEM/pyglpk/Data/gem_input_data.pkl
Contents:
  metadata             → shape (18, 16)
  nmr                  → shape (18, 38)
  microbiome           → shape (18, 44)
  pathways             → shape (18, 250)
  mean_expr            → shape (23017, 11)
  mean_expr_high       → shape (23017, 11)
  mean_expr_low        → shape (23017, 11)
  metabolite_map       → <class 'dict'>
  gem_metabolites      → <class 'list'>
  common_ids           → <class 'list'>


### Final Validation to make sure i have the correct data

In [14]:
# ============================================================
# 6: FINAL CHECK — EXACT DATA AVAILABLE FOR EACH MODALITY
# ============================================================

print("\n" + "="*70)
print("FINAL GEM INPUT CHECK")
print("="*70)

# ------------------------------------------------------------
# 1. SAMPLE ALIGNMENT
# ------------------------------------------------------------
print("\n[1] SAMPLE ALIGNMENT")
print(f"  Shared sample IDs (n={len(common_ids)}):")
print(" ", common_ids)

for name, df in {
    "metadata": metadata,
    "nmr": nmr,
    "microbiome": microbiome,
    "pathways": pathways_from_master
}.items():
    print(f"  {name:12s} → shape {df.shape}, index matches shared: {set(df.index)==set(common_ids)}")

# ------------------------------------------------------------
# 2. MODALITY CONTENT SUMMARY
# ------------------------------------------------------------
print("\n[2] MODALITY CONTENT SUMMARY")

def modality_summary(name, df):
    print(f"\n{name}")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {len(df.columns)}")
    print(f"  Missing values: {df.isna().sum().sum()}")
    if df.shape[1] < 20:
        print(f"  Column names: {list(df.columns)}")
    else:
        print(f"  First 5 columns: {list(df.columns[:5])}")

modality_summary("Metadata", metadata)
modality_summary("NMR metabolites", nmr)
modality_summary("Microbiome species", microbiome)
modality_summary("Pathways", pathways_from_master)

# ------------------------------------------------------------
# 3. SINGLE-CELL EXPRESSION SUMMARY
# ------------------------------------------------------------
print("\n[3] SINGLE-CELL EXPRESSION")

for name, df in {
    "mean_expr": mean_expr,
    "mean_expr_high": mean_expr_high,
    "mean_expr_low": mean_expr_low
}.items():
    print(f"\n{name}")
    print(f"  Shape: {df.shape}")
    print(f"  Genes: {df.shape[0]}, Cell types: {df.shape[1]}")
    print(f"  Value range: {df.values.min():.3f} to {df.values.max():.3f}")
    print(f"  Missing values: {df.isna().sum().sum()}")

# ------------------------------------------------------------
# 4. METABOLITE MAPPING CHECK
# ------------------------------------------------------------
print("\n[4] METABOLITE MAPPING CHECK")

print("  Target metabolites (n=10):")
for m in gem_data["gem_metabolites"]:
    print("   -", m)

print("\n  Human-GEM IDs found:")
for k, v in metabolite_map.items():
    print(f"   {k:15s} → {v}")

missing = [m for m in gem_data["gem_metabolites"] if m not in metabolite_map]
if missing:
    print("\n  ⚠ Missing mappings:", missing)
else:
    print("\n  ✓ All 10 metabolites successfully mapped to Human-GEM IDs")

# ------------------------------------------------------------
# 5. CHECK THAT ALL 10 METABOLITES EXIST IN THE NMR MATRIX
# ------------------------------------------------------------
print("\n[5] CHECK NMR MATRIX FOR TARGET METABOLITES")

nmr_missing = [m for m in gem_data["gem_metabolites"] if m not in nmr.columns]
if nmr_missing:
    print("  ⚠ Missing from NMR:", nmr_missing)
else:
    print("  ✓ All 10 metabolites present in NMR data")

# ------------------------------------------------------------
# 6. FINAL READINESS
# ------------------------------------------------------------
print("\n" + "="*70)
print("GEM INPUT STATUS")
print("="*70)

if (
    len(common_ids) == 18
    and not missing
    and not nmr_missing
):
    print("✓ All modalities aligned")
    print("✓ All 10 metabolites mapped")
    print("✓ All 10 metabolites present in NMR")
    print("✓ Data is READY for GEM modelling")
else:
    print("⚠ Something needs fixing before GEM modelling")



FINAL GEM INPUT CHECK

[1] SAMPLE ALIGNMENT
  Shared sample IDs (n=18):
  ['GYECZ1177.LACF.LES', 'GYECZ1178.LPF.LES', 'GYECZ1178.LPF.NONLES', 'GYECZ1183.LACF.LES', 'GYECZ1183.LACF.NONLES', 'GYECZ1187.RACF.LES', 'GYECZ1187.RACF.NONLES', 'GYECZ1190.BACK.LES', 'GYECZ1190.LACF.LES', 'GYECZ1191.ABDO.LES', 'GYECZ1191.ABDO.NONLES', 'GYECZ1191.LACF.LES', 'GYECZ1193.LACF.LES', 'GYECZ1193.LACF.NONLES', 'GYECZ1197.BACK.LES', 'GYECZ1197.RACF.NONLES', 'GYECZ1199.BACK.NONLES', 'GYECZ1199.RACF.LES']
  metadata     → shape (18, 16), index matches shared: True
  nmr          → shape (18, 38), index matches shared: True
  microbiome   → shape (18, 44), index matches shared: True
  pathways     → shape (18, 250), index matches shared: True

[2] MODALITY CONTENT SUMMARY

Metadata
  Shape: (18, 16)
  Columns: 16
  Missing values: 2
  Column names: ['eg_id', 'sample_short', 'sample_id', 'sa_abundance', 'SA_status', 'pH_mean', 'EASI', 'TEWL_mean', 'lesional', 'sample_site', 'age', 'sex', 'weeks_since_last_T